### Computing differences between immutable data states for results reproducibility analysis

This tutorial introduces the `state-diff` software designed to enable low-overhead and scalable analysis of intermediate results produced by scientific applications. We will introduce different APIs exposed to transform and compare intermediate results (captured through checkpointing) to identify whether the result from the current application run are within acceptable margin of a previous/reference run.

![Alt text](statediff-workflow.png "Statediff Workflow")

In this tutorial, we randomly generate two floating-point vectors representing checkpoints captured during two runs of an application and analysis their differences in accordance with state-diff's workflow depicted in the figure above.

In [1]:
import numpy as np

# Parameters
ckpt_size = 10**6  # Number of elements in the array
low_bound = 0.0     # Lower bound for random float values
high_bound = 100.0  # Upper bound for random float values
ckpt1_fname = "data/run1_ckpt.dat"
ckpt2_fname = "data/run2_ckpt.dat"

# Random checkpoints generation
ckpt1_data = np.random.uniform(low=low_bound, high=high_bound, size=ckpt_size)
ckpt2_data = np.random.uniform(low=low_bound, high=high_bound, size=ckpt_size)

# Save checkpoints
with open(ckpt1_fname, 'wb') as f1:
    ckpt1_data.tofile(f1)

with open(ckpt2_fname, 'wb') as f2:
    ckpt2_data.tofile(f2)

<u>**Note**:</u> Before running the example below, ensure state-diff is build and installed by executing :```bash ../../builds/state-diff/build.sh```
A first-time installation takes several minutes due to spack packages installation. However, spack packages are cached, completiong subsequent installations in approximatelly 2 minutes.

In [2]:
# !source statediff-RECUP/venv/bin/activate

#### Import reader and client tools from statediff

In [ ]:
from statediff import IOUringStream, StateDiffClient

#### Define parameters for results reproducibility analysis

In [ ]:
# Size (bytes) of data to analyse
data_size = 1024 * 1024
# size of data chunk representing a leaf in the generated compact data representation
chunk_size = 512
# Error tolerance within which collected results are considered reproducible
error_tol = 1e-4
# Data type
dtype = 'f'
# Root level determines the level of the tree from which statediff starts comparisons
root_level = 1

#### Instatiating data readers

In [ ]:
run1_reader = IOUringStream(ckpt1_fname, chunk_size // 4)
run2_reader = IOUringStream(ckpt2_fname, chunk_size // 4)

#### Instatiating statediff clients

In [ ]:
run1_client = StateDiffClient(1, run1_reader, data_size, error_tol, dtype, chunk_size, root_level)
run2_client = StateDiffClient(2, run2_reader, data_size, error_tol, dtype, chunk_size, root_level)

#### Compaction of intermediate results into Merkle trees

In [ ]:
run1_client.create(run1_data)
run2_client.create(run2_data)

#### Comparison of intermediate results from two runs

In [ ]:
run1_client.compare_with(run2_client)

#### Analysis of comparison results and summary statistics

In [ ]:
num_different = run1_client.get_num_changes()
print(f"Number of different floating-point numbers: {num_different}")
if num_different > 0:
    print("Conclusion: Intermediate results are different!")
else:
    print("Conclusion: Intermediate results are within error margin!)